# Olist Seller Intelligence Report

This notebook presents an analysis of the [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data), a real-world e-commerce dataset covering orders placed between 2016 and 2018 across multiple Brazilian marketplaces.

The analysis is framed as a data consultancy project: Olist's commercial team wants to understand how sellers are performing, what's driving poor customer reviews, and where inefficiencies exist in their logistics. The goal is to produce findings and recommendations that could inform real operational decisions.

The dataset is stored in MongoDB Atlas and queried using PyMongo. Aggregation pipelines are the primary analytical tool, with results passed into Pandas dataframes for presentation and visualisation. The secondary aim of the notebook is to provide a working demonstration of MongoDB's aggregation framework, index optimisation, and Python driver usage.

# Section 1 - Setup and Data Quality Audit

The client has asked me to first verify that the data is complete and clean before any analysis. I also need to log the session and flag any known data issues for the record.

## 1.1 Connect to the database and list all collections

In [1]:
#import libraries
from pymongo import MongoClient
import pandas as pd
import os
from dotenv import load_dotenv
from pprint import pprint
import datetime
from utils import count_docs
import json

In [2]:
#load/declare constants
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
MISSING_CONDITIONS = [{"$in": [None, ""]}, {"$exists": False}]

In [3]:
#connect to database
client = MongoClient(MONGO_URI)
db = client["oilist-sales"]

In [4]:
#store collections as variable and view
collections = db.list_collection_names()
print(collections)

['data_audit', 'products', 'order-payments', 'geolocation', 'order-reviews', 'category-translation', 'order-items', 'customers', 'orders', 'sellers']


All collections are present as expected.

## 1.2. Inspect one document from each collection and note the schema

In order to understand the schema properly, I will not just inspect one document as the schema can vary from document-to-document, due to MongoDB's nature as a schemaless database. I will instead take a sample of 1,000 documents from each collection, and find the union of the fields that appear across them. This gives a much better chance to capture the most commonly-appearing fields for each collection, thereby producing a more reliable outline of the schema than inspecting just one document.

In [5]:
#get sample of 100 docs
for collection in collections:
    sample = db[collection].find({}, limit = 1000)
    #find all values present in sample
    fields = {}
    for doc in sample:
        for field, value in doc.items():
            if field not in fields and value is not None:
                fields[field] = type(value).__name__
    #display schema for each collection as df
    schema = pd.DataFrame(fields.items(), columns=["field", "type"])
    print(collection)
    display(schema)

data_audit


,field,type
0,_id,ObjectId
1,auditor,str
2,date,datetime
3,details,str


products


,field,type
0,_id,ObjectId
1,product_id,str
2,product_category_name,str
3,product_name_lenght,int
4,product_description_lenght,int
5,product_photos_qty,int
6,product_weight_g,int
7,product_length_cm,int
8,product_height_cm,int
9,product_width_cm,int


order-payments


...

,field,type
0,_id,ObjectId
1,customer_id,str
2,customer_unique_id,str
3,customer_zip_code_prefix,str
4,customer_city,str
5,customer_state,str


orders


,field,type
0,_id,ObjectId
1,order_id,str
2,customer_id,str
3,order_status,str
4,order_purchase_timestamp,str
5,order_approved_at,str
6,order_estimated_delivery_date,datetime
7,order_delivered_carrier_date,str
8,order_delivered_customer_date,str


sellers


,field,type
0,_id,ObjectId
1,seller_id,str
2,seller_zip_code_prefix,str
3,seller_city,str
4,seller_state,str


### 1.2.1 Correct spelling errors in field names

### 1.2.2 Correct value typing errors

## 1.3. Create a collection (with a validation schema) to record data audits

In [6]:
# db.create_collection("data_audit", validator = {"$jsonSchema": {
#     "required": ["auditor", "date", "details"],
#     "properties": {
#         "auditor": {"bsonType": "string"},
#         "date": {"bsonType": "date"},
#         "details": {"bsonType": "string"}
#     }
# }})

In [7]:
#add audit log document to collection
# db.data_audit.insert_one(
#     {
#         "auditor": "Liam Cottrell",
#         "date": datetime.datetime.now(),
#         "details": "Created data audit collection and viewed schemas for each collection"
#     }
# )

In [8]:
#view document to confirm successful creation
db["data_audit"].find_one()

{'_id': ObjectId('6a287866482bcd769a72ecd2'),
 'auditor': 'Liam Cottrell',
 'date': datetime.datetime(2026, 6, 9, 21, 32, 38, 894000),
 'details': 'Created data audit collection and viewed schemas for each collection'}

## 1.4 Count documents in each collection and present the results

In [19]:
collection_count = {}
for collection in collections:
    num_docs = db[collection].count_documents({})
    collection_count[collection] = num_docs
    print(f"The {collection} collection contains {num_docs} documents.")

The data_audit collection contains 1 documents.
The products collection contains 32951 documents.
The order-payments collection contains 103886 documents.
The geolocation collection contains 1000163 documents.
The order-reviews collection contains 99224 documents.
The category-translation collection contains 71 documents.
The order-items collection contains 112650 documents.
The customers collection contains 99441 documents.
The orders collection contains 99441 documents.
The sellers collection contains 3095 documents.


## 1.5 Identify orders with missing customer delivery dates

In [21]:
#call function to get missing dates
missing_dates_ct, missing_dates_pct = count_docs(db, "orders", "order_delivered_customer_date", "$or", MISSING_CONDITIONS)
dates_finding = f"There are {missing_dates_ct} orders without a delivery date as confirmed by the customer, representing {round(missing_dates_pct * 100, 2)}% of all orders."
print(dates_finding)

There are 2965 orders without a delivery date as confirmed by the customer, representing 2.98% of all orders.


## 1.6 Identify reviews with no comments

In [23]:
#call function to get missing comments
missing_comments_ct, missing_comments_pct = count_docs(db, "order-reviews", "review_comment_message", "$or", MISSING_CONDITIONS)
comments_finding = f"There are {missing_comments_ct} customer reviews without a comment, representing {round(missing_comments_pct * 100, 2)}% of all reviews."
print(comments_finding)

There are 58247 customer reviews without a comment, representing 58.7% of all reviews.


## 1.7 Remove products with missing category names

In [22]:
#call function to get missing categories
missing_cats_ct, missing_cats_pct = count_docs(db, "products", "product_category_name", "$or", MISSING_CONDITIONS)
cats_finding = f"There are {missing_cats_ct} products missing a category name, representing {round(missing_cats_pct * 100, 2)}% of all products."
print(cats_finding)

There are 610 products missing a category name, representing 1.85% of all products.


In [14]:
#remove products from database
# db["products"].delete_many({"$or": [{"product_category_name": {"$in": [None, ""]}}, {"product_category_name": {"$exists": False}}]})

## 1.8 Outline order status values

Next, the client is interested in finding any orders with a status other than "delivered" or "shipped". I will count these, and list the distinct statuses present in the dataset.

In [24]:
#call function to get unfilfilled orders
unfulfilled_statuses_ct, unfulfilled_statuses_pct = count_docs(db, "orders", "order_status", "$nor", [{"$eq": "shipped"}, {"$eq": "delivered"}])
unfulfilled_finding = f"There are {unfulfilled_statuses_ct} orders that are unfulfilled (not shipped or delivered), representing {round(unfulfilled_statuses_pct * 100, 2)}% of all orders."
print(unfulfilled_finding)

There are 1856 orders that are unfulfilled (not shipped or delivered), representing 1.87% of all orders.


In [18]:
#find unique order status values
statuses = db["orders"].distinct("order_status")
print(statuses)

['approved', 'canceled', 'created', 'delivered', 'invoiced', 'processing', 'shipped', 'unavailable']


## 1.9 Create a collection for logistics partners and insert data

The client flagged that courier data had been omitted from the original dataset. They've provided a JSON file containing details of the logistics partners used for fulfilment, which will be needed later in the project. I will import this data into a new collection.

In [39]:
#import logistics partners from json file
logistics_partners = json.loads(open("logistics-partners.json").read())
#print to check
pprint(logistics_partners[0])
print(f"There are {len(logistics_partners)} logistics partners.")

{'active': True,
 'avg_delivery_days': 8,
 'carrier_code': 'COR',
 'carrier_name': 'Correios',
 'on_time_rate': 0.71,
 'regions_covered': ['SP',
                     'RJ',
                     'MG',
                     'BA',
                     'PR',
                     'RS',
                     'SC',
                     'PE',
                     'CE',
                     'GO']}
There are 6 logistics partners.


In [31]:
#create and validate new collection
db.create_collection("logistics_partners", validator = {"$jsonSchema": {
    "required": ["carrier_name", "carrier_code", "regions_covered", "avg_delivery_days", "on_time_rate", "active"],
    "properties": {
        "carrier_name": {"bsonType": "string"},
        "carrier_code": {"bsonType": "string"},
        "regions_covered": {"bsonType": "array"},
        "avg_delivery_days": {"bsonType": "int"},
        "on_time_rate": {"bsonType": "double"},
        "active": {"bsonType": "bool"}
    }
}})

Collection(Database(MongoClient(host=['ac-vf0ykm7-shard-00-01.musfbjl.mongodb.net:27017', 'ac-vf0ykm7-shard-00-00.musfbjl.mongodb.net:27017', 'ac-vf0ykm7-shard-00-02.musfbjl.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='liamscluster', authsource='admin', replicaset='atlas-1435mb-shard-0', tls=True), 'oilist-sales'), 'logistics_partners')

In [40]:
db.logistics_partners.insert_many(logistics_partners)

InsertManyResult([ObjectId('6a38343ae62d4cd7609b6dd9'), ObjectId('6a38343ae62d4cd7609b6dda'), ObjectId('6a38343ae62d4cd7609b6ddb'), ObjectId('6a38343ae62d4cd7609b6ddc'), ObjectId('6a38343ae62d4cd7609b6ddd'), ObjectId('6a38343ae62d4cd7609b6dde')], acknowledged=True)

In [44]:
#view doc to check insert
pprint(db["logistics_partners"].find_one())
print(f"There are {db['logistics_partners'].count_documents({})} logistics partners.")

{'_id': ObjectId('6a38343ae62d4cd7609b6dd9'),
 'active': True,
 'avg_delivery_days': 8,
 'carrier_code': 'COR',
 'carrier_name': 'Correios',
 'on_time_rate': 0.71,
 'regions_covered': ['SP',
                     'RJ',
                     'MG',
                     'BA',
                     'PR',
                     'RS',
                     'SC',
                     'PE',
                     'CE',
                     'GO']}
There are 6 logistics partners.


## 1.10 Update audit log with findings

In [ ]:
# #add new log
# db.data_audit.insert_one(
#     {
#         "auditor": "Liam Cottrell",
#         "date": datetime.datetime.now(),
#         "details": "TBC"
#     }
# )